In [3]:
%pip install -q -U transformers accelerate

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

In [7]:
%pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 55.4 MB/s  0:00:00eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import time
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

assert torch.cuda.is_available(), "Necesita GPU"

In [10]:
carpeta = next(c for c in ["/workspace", "/content", "../../reseñas"]
               if os.path.exists(f"{c}/resenas_espanol_trad.csv"))

df_es = pd.read_csv(f"{carpeta}/resenas_espanol_trad.csv")
df_pt = pd.read_csv(f"{carpeta}/resenas_portugues_trad.csv")

In [11]:
# Muestra
muestra_es = df_es.sample(n=500, random_state=42).copy()
muestra_pt = df_pt.sample(n=500, random_state=42).copy()

In [12]:
# Traducción de la muestra
def traducir_muestra(df, columna, idioma, funcion):
    traducciones = []
    tiempos = []

    for texto in tqdm(df[columna].astype(str)):
        traduccion, tiempo = funcion(texto, idioma)
        traducciones.append(traduccion)
        tiempos.append(tiempo)

    return traducciones, tiempos

In [13]:
# Hy-MT2: carga del modelo
modelo_id = "tencent/Hy-MT2-1.8B"

tokenizer = AutoTokenizer.from_pretrained(modelo_id, trust_remote_code=True)
modelo = AutoModelForCausalLM.from_pretrained(
    modelo_id, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
).eval()

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'beta_slow', 'beta_fast', 'mscale', 'alpha', 'mscale_all_dim'}


tokenizer_config.json:   0%|          | 0.00/166k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.53M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/488 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/654 [00:00<?, ?B/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='dynamic': {'beta_slow', 'beta_fast', 'mscale', 'alpha', 'mscale_all_dim'}


model.safetensors: reconstructing file:   0%|          |  0.00B / 4.08GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/221 [00:00<?, ?B/s]

In [14]:
# Hy-MT2: traducción
def traducir_hymt(texto, idioma):
    prompt = (
        "Translate the following text into English. Note that you should "
        "only output the translated result without any additional explanation:\n\n" + texto
    )
    mensajes = [{"role": "user", "content": prompt}]

    # tiempo de punta a punta (tokenizar + generar + decodificar), igual que el pipeline de translategemma
    torch.cuda.synchronize()
    inicio = time.perf_counter()

    inputs = tokenizer.apply_chat_template(
        mensajes, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(modelo.device)

    with torch.no_grad():
        # parámetros recomendados
        outputs = modelo.generate(
            **inputs, max_new_tokens=1024,
            do_sample=True, temperature=0.7, top_p=0.6, top_k=20, repetition_penalty=1.05,
        )

    # solo la traducción
    nuevos = outputs[0][inputs["input_ids"].shape[-1]:]
    traduccion = tokenizer.decode(nuevos, skip_special_tokens=True).strip()

    torch.cuda.synchronize()
    tiempo = time.perf_counter() - inicio

    return traduccion, tiempo

In [15]:
# Prueba
traducir_hymt("La comida estaba deliciosa, pero la atención fue muy lenta.", "es")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


('The food was delicious, but the service was very slow.', 1.1841420289129019)

In [16]:
# Hy-MT2 español
muestra_es["hymt"], muestra_es["tiempo_hymt"] = traducir_muestra(muestra_es, "Review Text_es", "es", traducir_hymt)

  0%|          | 0/500 [00:00<?, ?it/s]

In [17]:
# Hy-MT2 portugués
muestra_pt["hymt"], muestra_pt["tiempo_hymt"] = traducir_muestra(muestra_pt, "Review Text_pt-BR", "pt", traducir_hymt)

  0%|          | 0/500 [00:00<?, ?it/s]

In [18]:
# Memoria y liberar GPU
memoria_hymt = torch.cuda.max_memory_allocated() / 1e9
print(f"Memoria GPU Hy-MT2: {memoria_hymt:.1f} GB")

del modelo, tokenizer
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

Memoria GPU Hy-MT2: 4.3 GB


In [19]:
# Google translategemma: carga del modelo
pipe = pipeline(
    "image-text-to-text",
    model="google/translategemma-4b-it",
    device="cuda",
    dtype=torch.bfloat16,
)

# códigos de idioma
codigos = {"es": "es", "pt": "pt-BR"}

config.json:   0%|          | 0.00/2.62k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [20]:
def traducir_gemma(texto, idioma):
    mensaje = [{
        "role": "user",
        "content": [{
            "type": "text",
            "source_lang_code": codigos[idioma],
            "target_lang_code": "en",
            "text": texto,
        }],
    }]

    # tiempo
    torch.cuda.synchronize()
    inicio = time.perf_counter()

    out = pipe(text=[mensaje], max_new_tokens=1024)

    torch.cuda.synchronize()
    tiempo = time.perf_counter() - inicio

    return out[0][0]["generated_text"][-1]["content"].strip(), tiempo

In [21]:
# Prueba
traducir_gemma("La comida estaba deliciosa, pero la atención fue muy lenta.", "es")

('The food was delicious, but the service was very slow.', 0.9390376824885607)

In [22]:
# Google español
muestra_es["gemma"], muestra_es["tiempo_gemma"] = traducir_muestra(muestra_es, "Review Text_es", "es", traducir_gemma)

  0%|          | 0/500 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [23]:
# Google portugués
muestra_pt["gemma"], muestra_pt["tiempo_gemma"] = traducir_muestra(muestra_pt, "Review Text_pt-BR", "pt", traducir_gemma)

  0%|          | 0/500 [00:00<?, ?it/s]

In [24]:
# liberar GPU
memoria_gemma = torch.cuda.max_memory_allocated() / 1e9
print(f"Memoria GPU translategemma: {memoria_gemma:.1f} GB")

del pipe
torch.cuda.empty_cache()

Memoria GPU translategemma: 8.9 GB


In [25]:
# Guardar traducciones, tiempos, memoria y GPU usada 
import json

muestra_es.to_csv(f"{carpeta}/comparacion_es.csv", index=False)
muestra_pt.to_csv(f"{carpeta}/comparacion_pt.csv", index=False)

with open(f"{carpeta}/info_gpu.json", "w") as f:
    json.dump({
        "gpu": torch.cuda.get_device_name(0),
        "memoria_hymt": memoria_hymt,
        "memoria_gemma": memoria_gemma,
    }, f)

In [26]:
%pip install -q bert-score unbabel-comet "setuptools<81"

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import json
import pandas as pd
import torch
from bert_score import score
from comet import download_model, load_from_checkpoint

carpeta = next(c for c in ["/workspace", "/content", "../../reseñas"]
               if os.path.exists(f"{c}/comparacion_es.csv"))

muestra_es = pd.read_csv(f"{carpeta}/comparacion_es.csv")
muestra_pt = pd.read_csv(f"{carpeta}/comparacion_pt.csv")

with open(f"{carpeta}/info_gpu.json") as f:
    info = json.load(f)
memoria_hymt, memoria_gemma = info["memoria_hymt"], info["memoria_gemma"]

/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [7]:
# BERTScore: traducción al inglés vs reseña original en inglés
def bertscore_f1(traducciones, originales):
    P, R, F1 = score(
        traducciones.tolist(),
        originales.tolist(),
        lang="en",                   
        rescale_with_baseline=True,  # 0 = no relacionado, 1 = idéntico
        device="cuda",
    )
    return F1.numpy()

In [8]:
# Prueba BERTScore: misma reseña original, distintas traducciones
original = "The ice cream was delicious, but the service was very slow."
candidatas = [
    "The ice cream was delicious, but the service was very slow.",    # idéntica
    "The ice cream was tasty, although the service was really slow.", # paráfrasis
    "The ice cream was delicious and the service was very fast.",     # sentido opuesto
    "I went to the park with my dog yesterday.",                      # nada que ver
]

f1 = bertscore_f1(pd.Series(candidatas), pd.Series([original] * len(candidatas)))
pd.DataFrame({"traducción": candidatas, "BERTScore F1": f1.round(3)})

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/bert_score/score.py:149: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  baselines = torch.from_numpy(


,traducción,BERTScore F1
0,"The ice cream was delicious, but the service w...",1.000
1,"The ice cream was tasty, although the service ...",0.920
2,The ice cream was delicious and the service wa...,0.876
3,I went to the park with my dog yesterday.,0.219


In [9]:
# BERTScore por modelo (referencia: reseña original en inglés)
muestra_es["f1_hymt"] = bertscore_f1(muestra_es["hymt"], muestra_es["Review Text"])
muestra_es["f1_gemma"] = bertscore_f1(muestra_es["gemma"], muestra_es["Review Text"])

muestra_pt["f1_hymt"] = bertscore_f1(muestra_pt["hymt"], muestra_pt["Review Text"])
muestra_pt["f1_gemma"] = bertscore_f1(muestra_pt["gemma"], muestra_pt["Review Text"])

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

In [10]:
# carga del modelo
kiwi = load_from_checkpoint(download_model("Unbabel/wmt22-cometkiwi-da"))

def cometkiwi(textos, traducciones):
    datos = [{"src": s, "mt": t} for s, t in zip(textos.astype(str), traducciones.astype(str))]
    return kiwi.predict(datos, batch_size=16, gpus=1, progress_bar=False).scores

.gitattributes: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

hparams.yaml:   0%|          | 0.00/716 [00:00<?, ?B/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.2 to v2.6.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint .cache/huggingface/hub/models--Unbabel--wmt22-cometkiwi-da/snapshots/1ad785194e391eebc6c53e2d0776cada8f83179a/checkpoints/model.ckpt`


sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/513 [00:00<?, ?B/s]

Encoder model frozen.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:216: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [11]:
# Prueba COMET-Kiwi: reseña en español, distintas traducciones al inglés
fuente = "El helado estaba delicioso, pero la atención fue muy lenta."
candidatas = [
    "The ice cream was delicious, but the service was very slow.",    # correcta
    "The ice cream was tasty, although the service was really slow.", # paráfrasis
    "The ice cream was delicious and the service was very fast.",     # sentido opuesto
    "I went to the park with my dog yesterday.",                      # nada que ver
]

kiwi_prueba = cometkiwi(pd.Series([fuente] * len(candidatas)), pd.Series(candidatas))
pd.DataFrame({"traducción": candidatas, "COMET-Kiwi": [round(k, 3) for k in kiwi_prueba]})

/usr/local/lib/python3.12/dist-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /usr/local/lib/python3.12/dist-packages/ipykernel_la ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA RTX 4000 Ada Generation') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_prec

,traducción,COMET-Kiwi
0,"The ice cream was delicious, but the service w...",0.873
1,"The ice cream was tasty, although the service ...",0.860
2,The ice cream was delicious and the service wa...,0.737
3,I went to the park with my dog yesterday.,0.335


In [12]:
# COMET-Kiwi por modelo (sin referencia)
muestra_es["kiwi_hymt"] = cometkiwi(muestra_es["Review Text_es"], muestra_es["hymt"])
muestra_es["kiwi_gemma"] = cometkiwi(muestra_es["Review Text_es"], muestra_es["gemma"])

muestra_pt["kiwi_hymt"] = cometkiwi(muestra_pt["Review Text_pt-BR"], muestra_pt["hymt"])
muestra_pt["kiwi_gemma"] = cometkiwi(muestra_pt["Review Text_pt-BR"], muestra_pt["gemma"])

/usr/local/lib/python3.12/dist-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /usr/local/lib/python3.12/dist-packages/ipykernel_la ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is a

In [13]:
# Resumen
resumen = pd.DataFrame({
    "BERTScore español": [muestra_es["f1_hymt"].mean(), muestra_es["f1_gemma"].mean()],
    "BERTScore portugués": [muestra_pt["f1_hymt"].mean(), muestra_pt["f1_gemma"].mean()],
    "COMET-Kiwi español": [muestra_es["kiwi_hymt"].mean(), muestra_es["kiwi_gemma"].mean()],
    "COMET-Kiwi portugués": [muestra_pt["kiwi_hymt"].mean(), muestra_pt["kiwi_gemma"].mean()],
    "segundos por reseña español": [muestra_es["tiempo_hymt"].mean(), muestra_es["tiempo_gemma"].mean()],
    "segundos por reseña portugués": [muestra_pt["tiempo_hymt"].mean(), muestra_pt["tiempo_gemma"].mean()],
    "memoria GPU (GB)": [memoria_hymt, memoria_gemma],
}, index=["Hy-MT2-1.8B", "translategemma-4b"])

resumen.round(3)

,BERTScore español,BERTScore portugués,COMET-Kiwi español,COMET-Kiwi portugués,segundos por reseña español,segundos por reseña portugués,memoria GPU (GB)
Hy-MT2-1.8B,0.628,0.643,0.820,0.808,4.304,4.196,4.325
translategemma-4b,0.669,0.676,0.821,0.806,6.409,6.270,8.870


In [14]:
# Latencia por reseña 
def latencia(df, modelo):
    t = df[f"tiempo_{modelo}"]
    palabras = df[modelo].astype(str).str.split().str.len()
    palabras = palabras.where(palabras > 0)
    return {
        "media": t.mean(),
        "mediana": t.median(),
        "p95": t.quantile(0.95),
        "máximo": t.max(),
        "reseñas por minuto": 60 / t.mean(),
        "ms por palabra traducida": (t / palabras).median() * 1000,
    }

filas = {}
for idioma, df in [("español", muestra_es), ("portugués", muestra_pt)]:
    for modelo, nombre in [("hymt", "Hy-MT2-1.8B"), ("gemma", "translategemma-4b")]:
        filas[(nombre, idioma)] = latencia(df, modelo)

print("GPU:", info["gpu"])
pd.DataFrame(filas).T.round(3)

GPU: NVIDIA RTX 4000 Ada Generation


,,media,mediana,p95,máximo,reseñas por minuto,ms por palabra traducida
Hy-MT2-1.8B,español,4.304,3.366,10.956,22.858,13.940,46.205
translategemma-4b,español,6.409,4.949,16.663,34.772,9.362,67.570
Hy-MT2-1.8B,portugués,4.196,3.288,10.226,31.967,14.301,46.091
translategemma-4b,portugués,6.270,4.928,15.169,51.601,9.569,67.178


In [15]:
# Reseñas para revisar
pd.set_option("display.max_colwidth", None)

revisar_es = muestra_es.sample(n=10, random_state=42)
revisar_pt = muestra_pt.sample(n=10, random_state=42)

In [16]:
# Revisión español
revisar_es[["Review Text", "Review Text_es", "hymt", "gemma", "f1_hymt", "f1_gemma", "kiwi_hymt", "kiwi_gemma"]]

,Review Text,Review Text_es,hymt,gemma,f1_hymt,f1_gemma,kiwi_hymt,kiwi_gemma
361,"Sugar overload! Just had a glazed donut, it was delicious, soft yet satisfyingly doughy at the same time, with a nice amount of glaze. As a first timer it was a bit overwhelming to have to keep asking what each donut was, but hang in there, and order what looks good for your dozen to take home. They have serve yourself to-go coffee, and assorted bottled beverages. Cash only, although their sign states they will accept a check with proper ID.","¡Demasiada azúcar! Acabo de comer un donut con glaseado, estaba delicioso, suave pero a la vez satisfactoriamente denso, con una buena cantidad de glaseado. Para alguien que lo probaba por primera vez, fue un poco abrumador tener que preguntar constantemente qué era cada tipo de donut, pero ¡mantente firme y pide los que te parezcan buenos para llevar a casa! Ofrecen café para llevar, y una variedad de bebidas embotelladas. Solo se aceptan pagos en efectivo, aunque en la señal indican que aceptan cheques con identificación válida.","Too much sugar! I just ate a donut with icing—it was delicious, soft yet satisfyingly dense, with plenty of icing. For someone trying it for the first time, it was a bit overwhelming to keep asking what each type of donut was. But stay persistent and ask for those you think are good to take home! They offer coffee to go, as well as a variety of bottled drinks. Only cash payments are accepted, although they indicate that they also accept checks with a valid ID.","""Too much sugar!"" I just ate a donut with icing, it was delicious – soft but also satisfyingly dense, with a good amount of icing. For someone trying it for the first time, it was a little overwhelming to have to constantly ask what each type of donut was, but stick to your guns and order the ones that look good to take home! They offer coffee to go, and a variety of bottled drinks. They only accept cash payments, although the sign indicates that they also accept checks with valid identification.",0.572012,0.613430,0.837584,0.831976
73,I'm addicted to the Sfingi's!!! Absolutely love them!! This is the best bakery. Hands down! Everything that I've purchased from here has been delicious! I highly recommend this place. First taste and you'll be hooked.,"¡Estoy completamente enganchado a las ""sfingi""! ¡Me encantan! Es la mejor panadería. Sin duda. Todo lo que he comprado aquí ha sido delicioso. Recomiendo encarecidamente este lugar. Una vez que pruebes, te quedarás.","I’m completely obsessed with the “Sphinx” products! I love them! It’s the best bakery. Without a doubt. Everything I’ve bought here has been delicious. I highly recommend this place. Once you try it, you’ll stay.","I'm completely hooked on ""Sfingi""! I love them! It's the best bakery. Without a doubt. Everything I've bought here has been delicious. I highly recommend this place. Once you try it, you'll be hooked.",0.563877,0.728723,0.814726,0.833024
374,I had a sweet tooth after eating lunch at STIR. Just a heads up do not walk there - there is not a dine in option. Please drive your car because after you place your order you have to wait for it - then pick it up. You'll be eating in your car (which is not a bad thing) . Just a huge heads up!! I ordered the peach cobbler and cinnamon peach praline. It was $5 a piece on a special they had going on. It was yummy. Just a lot of cinnamon. Very sweet. The peach praline tasted just like the peach cobbler with cinnamon glazed pecans on top. I could of went without all of the cinnamon and sweetness but that's my taste buds. There are different food options located in the building not just peach cobbler!,"Después de comer en STIR, sentí un fuerte antojo de algo dulce. Solo quería avisarles que no hay opción para comer allí. Por favor, vengan en coche, ya que después de hacer su pedido tendrán que esperar a que lo preparen y luego recogerlo. Estarán comiendo dentro de su coche (lo cual no es un problema), ¡solo quer

In [17]:
# Revisión portugués
revisar_pt[["Review Text", "Review Text_pt-BR", "hymt", "gemma", "f1_hymt", "f1_gemma", "kiwi_hymt", "kiwi_gemma"]]

,Review Text,Review Text_pt-BR,hymt,gemma,f1_hymt,f1_gemma,kiwi_hymt,kiwi_gemma
361,"We had the pop tart donut, cremÃ© brÃ»lÃ©e and an Oreo donut. The place is super small, but the courtyard behind is cute and nice to sit in. We went around 1030 and they were already running out of donuts though, so I'd recommend you get there earlier than that!","Experimentamos o donut com cobertura de ""pop tart"", o de creme brûlée e um de Oreo. O local é bem pequeno, mas o pátio nos fundos é charmoso e agradável para sentar. Chegamos por volta das 10h30, mas já estavam ficando sem donuts, então eu recomendaria chegar mais cedo!","We tried the donuts with “pop tart” topping, those with cream brûlée, and ones with Oreo. The place is quite small, but the patio at the back is charming and nice for sitting. We arrived around 10:30, but the donuts were running out, so I would recommend arriving earlier!","We tried the donut with ""Pop-Tart"" icing, the crème brûlée donut, and one with Oreo filling. The place is quite small, but the patio at the back is charming and pleasant for sitting. We arrived around 10:30, but they were already running out of donuts, so I would recommend arriving earlier!",0.541914,0.553810,0.826733,0.826728
73,"Dang, this place is yummy! After two visits to Mason's Creamery, I've decided that the best thing about them is that they have really awesome and unique ice cream flavors. Ranging from black sesame, ube, mocha oreo, lavender lemon to strawberry, cookies & cream, and Thai tea, the rotating flavors give you something exciting to look forward to trying. The ice cream is creamy and smooth. They mostly do 'hard' serve (but I wouldn't call it hard, like I said, it's so smooth!) and one or two soft serve flavors. They also have churros, milkshakes, and egg waffle cones on their menu. I've heard that they also do Ramen, but because of stupid Covid, I don't believe that is currently true. The staff here is friendly and I really like the vibe of this place. It's located on Bridge Street real close to Lorain Ave. Park on the street, grab your treats and either take them to go or have a seat outside behind the building.","Nossa, este lugar é delicioso! Depois de duas visitas à Mason's Creamery, decidi que o melhor daquele lugar é a variedade incrível e original de sabores de sorvete. Desde o sorvete de gergelim preto, ube, mocha Oreo, limão e lavanda, até morango, biscoitos e creme, e chá tailandês, os sabores variados oferecem algo interessante para experimentar. O sorvete é cremoso e suave. Eles oferecem principalmente sorvete ""normal"" (mas eu não chamaria de ""normal"", como disse, é muito macio!), e alguns sabores de sorvete cremoso. Eles também têm churros, milkshakes e cones de waffle com ovo no cardápio. Ouvi dizer que também servem Ramen, mas, devido ao Covid, não acredito que isso seja verdade atualmente. A equipe aqui é simpática e eu realmente gosto do ambiente deste lugar. Ele fica na Bridge Street, bem perto da Lorain Ave. Estacione na rua, pegue seus doces e leve-os para viagem ou sente-se ao ar livre atrás do prédio.","Wow, this place is delicious! After visiting Mason’s Creamery twice, I decided that the best thing about that place is the incredible and unique variety of ice cream flavors. From black sesame, blueberry, mocha Oreo, lemon and lavender ice creams, to strawberries, cookies and cream, and Thai tea, the diverse flavors offer something interesting to try. The ice creams are creamy and smooth. They mainly offer “regular” ice creams (but I wouldn’t call them regular—they’re very soft!), as well as some creamy ice cream flavors. They also have churros, milkshakes, and waffle cones with eggs on the menu. I heard they serve ramen too, but due to COVID, I don’t think that’s true right now. The staff here is friendly, and I really enjoy the atmosphere at this place. It’s located on Bridge Street, right near Lorain Ave. Park your car in the street, pick up your treats, and take them with you, or sit outside 